# DeepVerify EXP-001 Held-Out Evaluation

This notebook provides the authoritative, reproducible workflow for the held-out test evaluation of the `EXP-001-spatial-baseline` model.

> **RESEARCH LIMITATIONS & SCOPE WARNING:**
> - The metrics and results generated by this evaluation are strictly benchmark-specific (140k Real and Fake Faces dataset).
> - High accuracy on this specific test split **does not** establish DeepVerify as a universal or generalized AI-image detection system.
> - Cross-dataset and cross-generator generalization (e.g., against Stable Diffusion or unseen GANs) remains formal future work and is unsupported by this baseline.

## 1. Global Evaluation Configuration
Define the immutable provenance hashes, experiment identifiers, and repository revision.

In [ ]:
# Evaluation Configuration
EXPECTED_REPO_REVISION = "50b6e5d1688db86111e39143ff07ae43fc7733de"
EXPERIMENT_ID = "EXP-001"
RUN_ID = "run_20261007_141902_9e04eeda"

# Cryptographic Hashes
EXPECTED_CHECKPOINT_SHA256 = "b05a302c9283783fdc7352abe967baa4e4ab278028a2ba08cf40bd290e521d03"
EXPECTED_MANIFEST_SHA256 = "f9bef275982fb9f6c8c58cbea4e5e2885019417a71324e4fe716fe39384de466"
EXPECTED_DATASET_ARCHIVE_SHA256 = "ddaec55d781628e9f356543aa05c4946c3e6670b9232998b538579332ce6929f"

# External Artifact Locations (Adjust based on your Drive structure)
DRIVE_ZIP_PATH = "/content/drive/MyDrive/DeepVerify/datasets/140k-real-and-fake-faces.zip"
DRIVE_RUN_ARTIFACTS_DIR = f"/content/drive/MyDrive/DeepVerify/experiments/{EXPERIMENT_ID}/{RUN_ID}"


## 2. Environment Setup & Authentication
Mount Google Drive and securely clone the DeepVerify repository using Colab Secrets.

In [ ]:
import os
import tempfile
import subprocess
import stat
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab, assuming local execution.")

REPO_ROOT = Path('/content/DeepVerify')

if os.path.exists(str(REPO_ROOT)):
    raise RuntimeError(
        "STOP. /content/DeepVerify already exists. "
        "Refusing to overwrite silently. Delete it manually if you intend to restart."
    )

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise ValueError("Empty token")
except Exception:
    raise RuntimeError(
        "STOP. GITHUB_TOKEN not found in Colab Secrets. "
        "Add your personal access token to Colab Secrets under the key GITHUB_TOKEN."
    )

ASKPASS_SCRIPT = """#!/bin/sh
case "$1" in
    Username*) printf '%s\\n' "$DEEPVERIFY_GITHUB_USERNAME" ;;
    Password*) printf '%s\\n' "$DEEPVERIFY_GITHUB_TOKEN" ;;
esac
"""

askpass_fd, askpass_path = tempfile.mkstemp(prefix='dv_askpass_', suffix='.sh')
env = os.environ.copy()
try:
    with os.fdopen(askpass_fd, 'w') as f:
        f.write(ASKPASS_SCRIPT)
    os.chmod(askpass_path, stat.S_IRUSR | stat.S_IXUSR)

    env['GIT_ASKPASS'] = askpass_path
    env['GIT_TERMINAL_PROMPT'] = '0'
    env['DEEPVERIFY_GITHUB_USERNAME'] = 'shairkhanfaizan'
    env['DEEPVERIFY_GITHUB_TOKEN'] = token

    repo_url = 'https://github.com/shairkhanfaizan/DeepVerify.git'
    result = subprocess.run(
        ['git', 'clone', repo_url, str(REPO_ROOT)],
        env=env,
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        raise RuntimeError("STOP. git clone failed. Verify GITHUB_TOKEN access.")

    fetch_result = subprocess.run(
        ['git', 'fetch', 'origin'],
        env=env,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True
    )
    
    checkout = subprocess.run(
        ['git', 'checkout', EXPECTED_REPO_REVISION],
        cwd=str(REPO_ROOT),
        capture_output=True, text=True
    )
    if checkout.returncode != 0:
        raise RuntimeError(f"STOP. git checkout {EXPECTED_REPO_REVISION} failed.")

    print(f"Successfully authenticated and checked out revision: {EXPECTED_REPO_REVISION}")
finally:
    if os.path.exists(askpass_path):
        os.remove(askpass_path)
    env.pop('DEEPVERIFY_GITHUB_TOKEN', None)
    env.pop('DEEPVERIFY_GITHUB_USERNAME', None)
    env.pop('GIT_ASKPASS', None)


In [ ]:
# Install Dependencies (Safely)
import sys
import subprocess

print("Verifying and installing missing dependencies safely...")
# Colab natively includes highly-optimized torch, torchvision, sklearn, numpy, pandas.
# We ONLY install pytest explicitly since it is required for unit tests.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest'], check=True)

import torch
import torchvision
import sklearn
import pytest
print(f"PyTorch version: {torch.__version__}")
print(f"torchvision version: {torchvision.__version__}")
print(f"sklearn version: {sklearn.__version__}")
print(f"pytest version: {pytest.__version__}")
print("Dependencies verified successfully without altering Colab optimizations.")


## 3. Artifact Restoration
Restore the precise frozen config and verified best model checkpoint.

In [ ]:
import shutil
import hashlib

run_dir = REPO_ROOT / f"experiments/runs/{EXPERIMENT_ID}/{RUN_ID}"
run_dir.mkdir(parents=True, exist_ok=True)

local_ckpt = run_dir / "model_best.pth"
local_config = run_dir / "config.yaml"
drive_ckpt = f"{DRIVE_RUN_ARTIFACTS_DIR}/model_best.pth"
drive_config = f"{DRIVE_RUN_ARTIFACTS_DIR}/config.yaml"

print(f"Restoring artifacts to {run_dir}...")
shutil.copy2(drive_ckpt, local_ckpt)
shutil.copy2(drive_config, local_config)

def compute_sha256(filepath):
    sha256_hash = hashlib.sha256()
    with open(filepath, "rb") as f:
        for byte_block in iter(lambda: f.read(4096 * 1024), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

actual_ckpt_sha = compute_sha256(local_ckpt)
if actual_ckpt_sha != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError(f"STOP. Checkpoint hash mismatch!\nExpected: {EXPECTED_CHECKPOINT_SHA256}\nGot: {actual_ckpt_sha}")

print(f"Artifacts restored and Checkpoint SHA-256 perfectly verified: {actual_ckpt_sha}")


## 4. Dataset Integrity & Extraction
Verify the zip file hash and perform the 140,000-file, manifest-driven secure extraction.

In [ ]:
if not os.path.exists(DRIVE_ZIP_PATH):
    raise FileNotFoundError(f"STOP. Dataset ZIP not found at {DRIVE_ZIP_PATH}")

print("Verifying dataset archive SHA-256...")
actual_zip_sha = compute_sha256(DRIVE_ZIP_PATH)
if actual_zip_sha != EXPECTED_DATASET_ARCHIVE_SHA256:
    raise RuntimeError(f"STOP. Archive SHA mismatch!\nExpected: {EXPECTED_DATASET_ARCHIVE_SHA256}\nGot: {actual_zip_sha}")
print("Dataset archive SHA-256 perfectly verified.")


In [ ]:
import zipfile
import csv
import yaml

# Ensure the manifest matches the config expectation
with open(local_config, "r") as f:
    config = yaml.safe_load(f)

manifest_path = REPO_ROOT / config["dataset"]["split_manifest"]
actual_manifest_sha = compute_sha256(manifest_path)
if actual_manifest_sha != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError(f"STOP. Manifest SHA mismatch!\nExpected: {EXPECTED_MANIFEST_SHA256}\nGot: {actual_manifest_sha}")
print("Frozen manifest SHA-256 perfectly verified.")

extract_dir_path = REPO_ROOT / "data" / "raw"
expected_140k_dir = extract_dir_path / "140k"

if expected_140k_dir.exists():
    print("Previous canonical extraction detected. Removing to ensure clean state...")
    shutil.rmtree(expected_140k_dir)

with tempfile.TemporaryDirectory() as tmpdir:
    tmp_extract_path = Path(tmpdir).resolve()
    print(f"Extracting ZIP to isolated temporary directory {tmp_extract_path}...")
    
    with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as z:
        for member in z.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise RuntimeError(f"STOP. Malicious path detected in ZIP: {member.filename}")
            target_path = (tmp_extract_path / member.filename).resolve()
            if tmp_extract_path not in target_path.parents and tmp_extract_path != target_path:
                raise RuntimeError(f"STOP. Path traversal attempt: {member.filename}")
        z.extractall(path=tmp_extract_path)
    print("Raw isolated extraction complete.")
    
    print("Indexing extracted files from temporary directory...")
    extracted_files = []
    for ext in ['.jpg', '.jpeg', '.png', '.webp']:
        extracted_files.extend(list(tmp_extract_path.rglob(f"*{ext}")))
    
    basename_to_src = {}
    for f in extracted_files:
        if f.name in basename_to_src:
            raise RuntimeError(f"STOP. Ambiguous filename detected in extracted ZIP: {f.name}")
        basename_to_src[f.name] = f
    
    print("Pass 1: Verifying SHA-256 for all 140,000 files...")
    moves_to_make = []
    missing = []
    verified_count = 0
    with open(manifest_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            target_filepath = REPO_ROOT / row["filepath"]
            basename = target_filepath.name
            expected_file_sha = row["sha256"]
            
            if basename not in basename_to_src:
                missing.append(basename)
                continue
                
            src_file = basename_to_src[basename]
            actual_sha = compute_sha256(src_file)
            
            if actual_sha != expected_file_sha:
                raise RuntimeError(f"STOP. Hash mismatch for {basename}:\nExpected: {expected_file_sha}\nGot: {actual_sha}")
            
            moves_to_make.append((src_file, target_filepath))
            verified_count += 1
            
            if verified_count % 10000 == 0:
                print(f"  Verified {verified_count} / 140000...")
    
    if missing:
        raise RuntimeError(f"STOP. {len(missing)} manifest files missing from ZIP. Examples: {missing[:5]}")
    
    if len(moves_to_make) != 140000:
        raise RuntimeError(f"STOP. Expected exactly 140,000 files to move, got {len(moves_to_make)}")
    
    print("Pass 1 Complete: All 140,000 files verified against canonical manifest hashes successfully.")
    
    print("Pass 2: Moving verified files to canonical layout...")
    (expected_140k_dir / "real").mkdir(parents=True, exist_ok=True)
    (expected_140k_dir / "fake").mkdir(parents=True, exist_ok=True)
    
    for src_file, target_filepath in moves_to_make:
        target_filepath.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(src_file), str(target_filepath))
    
    print("Deterministic normalization complete.")


## 5. Evaluator Preflight Checks
Execute the unit tests to guarantee the evaluator's integrity in this environment before running inference.

In [ ]:
import subprocess
print("Running Evaluator Test Suite...")
res = subprocess.run(
    ['python3', '-m', 'pytest', 'ml/tests/test_evaluate_exp001.py', '-v'],
    cwd=str(REPO_ROOT),
    env=dict(os.environ, PYTHONPATH="ml/scripts:ml")
)
if res.returncode != 0:
    raise RuntimeError("STOP. Evaluator tests failed.")
print("Evaluator tests passed successfully.")


## 6. Official Held-Out Test Evaluation
Run the evaluator script over the 21,000 held-out TEST split samples.

In [ ]:
print("Starting exact evaluation run...")
cmd = [
    sys.executable,
    "ml/scripts/evaluate_exp001.py",
    "--run-dir", str(run_dir),
    "--expected-sha256", EXPECTED_CHECKPOINT_SHA256
]

process = subprocess.Popen(
    cmd,
    cwd=str(REPO_ROOT),
    env=dict(os.environ, PYTHONPATH="ml/scripts:ml"),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end="", flush=True)

process.wait()
if process.returncode != 0:
    raise RuntimeError(f"Evaluation failed with exit code {process.returncode}")
print("Evaluation successfully finished.")


## 7. Artifact Persistence & Hash Generation
Generate metrics hashes and copy back to Drive for archival.

In [ ]:
import json

eval_dir = run_dir / "evaluation"
test_metrics_path = eval_dir / "test_metrics.json"
cm_path = eval_dir / "confusion_matrix.json"

print("--- Final Evaluation Metrics ---")
with open(test_metrics_path, "r") as f:
    print(json.dumps(json.load(f), indent=2))

print("\nCopying evaluation artifacts to external storage...")
drive_eval_dir = Path(DRIVE_RUN_ARTIFACTS_DIR) / "evaluation"
drive_eval_dir.mkdir(parents=True, exist_ok=True)

for fpath in eval_dir.glob("*.json"):
    shutil.copy2(fpath, drive_eval_dir / fpath.name)
    print(f"Backed up {fpath.name} (SHA-256: {compute_sha256(fpath)})")

print("\nEVALUATION COMPLETE AND SECURELY PRESERVED.")
